# Data Merging — Stage 5 Themes 02: Factor Inventory

## Input
- `Data/Data_Collection/Final/Stage_5_Model_Ready/01_unioned/{7 table names}.parquet` (used to establish cadence/level exactly)
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/{agg_means, agg_full_moments, panel}.parquet` (used to determine what actually survived)
- Optional legacy description sources: `Data/Data_Collection/Final/Stage_3_Model_Ready/combined_full_moments_factor_inventory.csv`, `combined_means_factor_inventory.csv`
- Optional legacy theme source: `Data/Data_Collection/Final/Stage_3_Model_Ready/themes/combined_full_moments_theme_assignment.csv`
- `lib.review` (`base_factor_map`)

## Purpose
Builds the **definitive, current-pipeline base-factor inventory** — regenerated directly from the final assembled data files rather than carried forward from any earlier pipeline stage, so that any factor removed anywhere upstream is guaranteed absent by construction rather than by manual bookkeeping. It then layers on descriptions and theme assignments recovered from an older inventory (where names can be matched), and finally produces a second, finer-grained inventory at the individual-column (moment) level. The notebook is explicitly documented as read-only except for the CSVs it writes.

This notebook runs as a sequence of largely independent blocks, several of which were clearly added iteratively (ad-hoc manual patches, theme-merging cells) rather than planned as a single linear script — the writeup below follows that structure.

## Block 1 — Base-Factor Inventory (`factor_inventory.csv`)

### Cell 1: Cadence and Level
Reads the seven `01_unioned/` tables directly to determine, per base factor, its cadence (daily/weekly/monthly) and level (stock/macro) — using the same logic as Stage 5 notebook 01: a factor is stock-level iff it has a `_cwmean` column in one of the full-moments tables; cadence is read from which means-level source table (`agg_market_daily_means`, `weekly_raw`, `agg_market_monthly_means`) the factor's column appears in. Both properties are explicitly noted as unrecoverable from the column name alone (`copper_monthly` happens to say so, `bank_credit` doesn't).

### Cell 2: What Actually Survived
Reads column lists from the three final `02_assembled/` tables — post-union, post-identity-enforcement, post-manual-drop — and computes each dataset's base-factor set via `rv.base_factor_map`. Since these are the truly final files, any factor removed by *any* upstream rule cannot appear here. Checks that `panel` and `agg_means` hold an identical base-factor set (as they should, given the Stage 4 identity-enforcement pass), flagging any asymmetry as a sign something changed upstream since the last full run.

### Cell 3: Assemble the Inventory Table
Builds one row per base factor with: cadence, level, source table, whether it's one of the five binary regime indicators, presence flags for each of the three final datasets, count of moment columns actually present in the full-moments table, the specific moment suffixes present, and an `old_name` field reconstructing what this factor would have been called under the previous pipeline's naming convention (`monthly_{base}` for monthly factors, unprefixed otherwise) — this reconstructed name is what enables the description-merging step that follows.

### Cell 4: Merge Descriptions from an Older Inventory
Tries each path in `DESC_CANDIDATES` in order; the first non-null description found per factor wins (later sources cannot overwrite an already-found description). The `normalise_old()` matching function tries, in sequence: the exact name, the name with a stripped `monthly_` prefix, then the name with a stripped moment suffix (`_cwmean`, `_cwstd`, etc.) — but **only** falls through to suffix-stripping if the exact and prefix-stripped forms both fail to match a known current base factor. This ordering is explicitly designed to avoid a specific trap: thirteen macro factors have names that legitimately end in a moment-suffix-like string (`bull_bear_spread`, `vix_term_spread`, etc.) but aren't actually moment columns — trying the exact name first means these are matched correctly before the suffix-stripping logic ever gets a chance to mangle them.

Also opportunistically carries through any other useful old columns (theme, subtheme, category, source, vendor, family, units, frequency) if present in the old file, without overwriting anything already set.

Reports match rate and prints the list of factors that ended up without any description — expected to be genuinely new-to-this-pipeline factors (the weekly `_diff` factors), the binary regime indicators, or names the normaliser genuinely couldn't map (flagged for manual review).

### Cell 5: Save
Reorders columns and writes `factor_inventory.csv`. The trailing printed notes are worth preserving as documentation:
- `in_agg_means`, `in_agg_full_moments`, and `in_panel` should all be `True` for every row (the union enforces an identical base-factor set) — any `False` signals upstream drift since the last full pipeline run.
- `n_moment_cols` is 1 for macro factors and up to 6 for stock factors (cap-weighted mean in the means table + 5 cross-sectional moments in the full-moments table); values below that ceiling mean individual moments were dropped from an otherwise-surviving factor (noted as having happened to 21 factors).
- The five binary regime indicators pass through un-z-scored and were never assessed by any exclusion rule, so they have no theme by design — a decision is deferred on whether to theme them or exclude them entirely from theming.

## Block 2 — Ad-Hoc Manual Patches to `factor_inventory.csv`
Three standalone cells that directly edit the already-saved `factor_inventory.csv` in place, run after inspecting the gaps left by Block 1:

1. **Manual description injection** for five factors that Block 1's automated matching couldn't find descriptions for: the four weekly Fed-release `_diff` factors (`bank_credit_diff`, `ci_loans_diff`, `fed_assets_diff`, `reserves_diff`) and `open_interest_diff` — all genuinely new constructs from the Stage 3 weekly rebuild with no equivalent in the old pipeline, so hand-written descriptions are supplied directly.
2. **Theme merge from the old full-moments theme assignment.** Reuses the same `normalise_old()`-style matching logic to map old theme/subtheme columns onto current base factors, explicitly **filtering out the old "Calendar & Regime" theme** before merging (since those factors were excluded from this pipeline entirely), then reports match count and prints the flagged unmatched factors for manual review — expected to correspond to the new `_diff` weekly factors plus the binary regime indicators.
3. **Manual theme assignment** for the five `_diff` factors left themeless by the merge: the four Fed/banking `_diff` factors go to `'Credit Conditions'`, and `open_interest_diff` goes to `'Volatility & Options'`.

## Block 3 — Moment-Level Factor Inventory (`moment_inventory_long.csv` / `_wide.csv`)
A separate, finer-grained inventory: **one row per surviving column**, not per base factor — so it shows exactly which of the five possible moments each stock factor actually retained.

### Cells 1–2: Read Schema and Classify Each Column
Reads column names via `pyarrow.parquet.read_schema` (schema-only, instant regardless of file size, and explicitly noted as reading identically from any full-moments parquet — `02_assembled` or any split file — since the column list doesn't vary, only the row content does). Classifies each feature column via `bmap = rv.base_factor_map(fm_cols)`: a column that maps to itself is a `raw_level` (either a genuine macro level, or one of the thirteen `_spread`-named macro factors that `base_factor_map` correctly leaves whole because none has a companion `_cwmean`); otherwise, the matched moment suffix is extracted.

### Cell 3: Completeness Check
For every stock-level base factor, computes which of the five possible moments (`cwmean`, `cwstd`, `cwskew`, `cwkurt`, `spread`) are actually present, and flags any factor with fewer than 5 — this is the source of the earlier-cited "21 factors with a missing moment" figure. Prints the full incomplete list with which specific moments are absent for each.

### Cell 4: Pivot to Wide Format
Reshapes the long per-column inventory into one row per base factor with a True/False column per moment type — described as "usually what you actually want for retheming," since it directly answers "does this factor's `cwstd` column exist in the final dataset?" without needing to filter the long form each time.

### Cell 5: Merge Descriptions
Joins in `description` and `cadence` from the Block 1 base-factor inventory (`factor_inventory.csv`), with a graceful fallback message if that file hasn't been built yet.

### Cell 6: Save
Writes both the long and wide forms. Trailing notes: `raw_level` should be `True` only for macro factors and the thirteen legitimately-`_spread`-named macro factors; for stock factors, `cwmean` should be `True` on literally every row since the union mechanism requires a surviving cap-weighted mean as the precondition for the base factor existing at all — any stock-level row with `cwmean=False` indicates upstream drift worth investigating before doing any further retheming work from this file. Also gives an expected total-feature-count reconciliation (291 macro raw levels + up to 5 moments × 288 stock factors = 1704 if every stock factor were complete, minus the 21 factors with a documented missing moment).

## Block 4 — Attach Themes to the Wide Inventory and Build a Summary
Loads both `moment_inventory_wide.csv` and `factor_inventory.csv`, pulls the `theme_name` column from the latter, merges it onto the wide inventory (dropping and re-merging defensively in case the cell is re-run), and saves the updated wide file back in place. Then extracts a focused 5-column subset (`base_factor`, `level`, `cadence`, `theme`, `description`) and saves it separately as `base_factor_theme_summary.csv` — a lightweight reference table intended for quick lookup rather than the full moment-level detail.

## Block 5 — Split the Summary by Theme
Loads `base_factor_theme_summary.csv`, fills any missing `theme` value with the placeholder `'Unassigned_Binary_Indicators'` (covering the five binary regime indicators, which were deliberately left themeless throughout), then writes one CSV per unique theme into a new `by_theme/` subdirectory. Theme names are sanitized for use as filenames (spaces and `&` replaced, all non-alphanumeric/underscore characters stripped) before constructing each output filename.

## Output
- `Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes/factor_inventory.csv` — one row per base factor, with cadence, level, presence flags, moment-completeness info, description, and theme (after Block 2's manual patches).
- `Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes/moment_inventory_long.csv` — one row per surviving column.
- `Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes/moment_inventory_wide.csv` — one row per base factor, moments as True/False columns, plus theme (after Block 4).
- `Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes/base_factor_theme_summary.csv` — condensed 5-column reference (`base_factor`, `level`, `cadence`, `theme`, `description`).
- `Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes/by_theme/{theme_name}_factor_inventory.csv` — one file per theme, split from the summary table.

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# FACTOR INVENTORY FOR THE CURRENT PIPELINE
#
# Regenerates the definitive base-factor list from the assembled data files, so
# every factor removed anywhere in the pipeline is absent by construction. Then
# optionally merges descriptions from an older inventory, handling the monthly_
# prefix and the moment suffixes.
#
# Suggested location: Code/Data_Merging/Stage_5_Themes/02_factor_inventory.ipynb
# Read-only apart from the one CSV it writes.
# ═══════════════════════════════════════════════════════════════════════════════

import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append('../..')
import lib.review as rv

pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 400)

ROOT     = Path('../../../Data/Data_Collection/Final')
UNIONED  = ROOT / 'Stage_5_Model_Ready' / '01_unioned'
ASSEMBLED= ROOT / 'Stage_5_Model_Ready' / '02_assembled'
OUT_DIR  = ROOT / 'Stage_5_Model_Ready' / '05_themes'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional description sources. Every path that exists is tried in order and the
# first non-null description per factor wins. Add or remove freely.
DESC_CANDIDATES = [
    ROOT / 'Stage_3_Model_Ready' / 'combined_full_moments_factor_inventory.csv',
    ROOT / 'Stage_3_Model_Ready' / 'combined_means_factor_inventory.csv',
]

MOMENTS  = ('_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread')
BINARIES = ['vix_above_20', 'vix_above_30', 'curve_inverted_2y10y',
            'curve_inverted_3m10y', 'credit_stress']

# meta columns in the assembled files
META = {
    'agg_means':        {'date', 'target_daily_return'},
    'agg_full_moments': {'date', 'target_daily_return'},
    'panel':            {'permno', 'date', 'dlyret', 'dlycap'},
}

# source tables in 01_unioned, used to establish cadence and level exactly
FULL_MOM = {'daily':   'agg_market_daily_full_moments',
            'monthly': 'agg_market_monthly_full_moments'}
MEANS    = {'daily':   'agg_market_daily_means',
            'weekly':  'weekly_raw',
            'monthly': 'agg_market_monthly_means'}
SKIP_COLS = {'date', 'target_daily_return', 'target_monthly_return'}


# ── CELL 1 : cadence and level, read from the source tables ──────────────────
# Neither is recoverable from a column name. copper_monthly happens to say so;
# bank_credit does not. The seven files in 01_unioned are already split by source
# table, so both are exact.
#
# A stock factor is exactly one carrying a _cwmean column in a full-moments
# table. Everything else in a means table is macro. The means tables use BARE
# base factor names, so this is a cross-reference between the two aggregate
# tables rather than a read of the name itself.

print('=' * 100)
print('CADENCE AND LEVEL')
print('=' * 100)

stock_cadence = {}
for freq, tag in FULL_MOM.items():
    cols = pd.read_parquet(UNIONED / f'{tag}.parquet').columns
    for b in {c[:-len('_cwmean')] for c in cols if c.endswith('_cwmean')}:
        stock_cadence[b] = freq

cadence, level, src_table = {}, {}, {}
for freq, tag in MEANS.items():
    cols = pd.read_parquet(UNIONED / f'{tag}.parquet').columns
    bmap = rv.base_factor_map(cols)
    for c in cols:
        if c in SKIP_COLS:
            continue
        b = bmap[c]
        cadence[b]   = freq
        level[b]     = 'stock' if b in stock_cadence else 'macro'
        src_table[b] = tag

print(f'  base factors catalogued : {len(cadence)}')
print('  cadence : ' + '   '.join(
    f'{k} {sum(v == k for v in cadence.values())}'
    for k in ('daily', 'weekly', 'monthly')))
print('  level   : ' + '   '.join(
    f'{k} {sum(v == k for v in level.values())}' for k in ('stock', 'macro')))


# ── CELL 2 : which factors are actually in the final datasets ────────────────
# Read from 02_assembled, which is post-union, post-identity-enforcement and
# post-manual-drop. Anything removed anywhere upstream cannot appear here.

print('\n' + '=' * 100)
print('BASE FACTORS PRESENT IN EACH FINAL DATASET')
print('=' * 100)

present, columns_of = {}, {}
for name, meta in META.items():
    cols = pd.read_parquet(ASSEMBLED / f'{name}.parquet').columns
    bmap = rv.base_factor_map(cols)
    feats = [c for c in cols if c not in meta]
    present[name] = {bmap[c] for c in feats}
    columns_of[name] = {}
    for c in feats:
        columns_of[name].setdefault(bmap[c], []).append(c)
    print(f'  {name:<18} {len(cols):>5} columns   {len(feats):>5} features   '
          f'{len(present[name]):>4} base factors')

all_base = sorted(set().union(*present.values()))
print(f'\n  union across all three: {len(all_base)}')

diff = (present['panel'] ^ present['agg_means'])
print(f'  base factors not common to panel and agg_means: {len(diff)}'
      + (f'   {sorted(diff)}' if diff else '   (identity enforced)'))


# ── CELL 3 : assemble the inventory ─────────────────────────────────────────
# old_name reconstructs the previous pipeline's column name so descriptions can
# be merged. That pipeline prefixed every monthly feature with monthly_, so a
# monthly base factor Tax appeared as monthly_Tax.

rows = []
for b in all_base:
    fm_cols = columns_of['agg_full_moments'].get(b, [])
    suffixes = sorted({s for s in MOMENTS
                       for c in fm_cols if c.endswith(s)})
    cad = cadence.get(b, '?')
    rows.append({
        'base_factor':        b,
        'cadence':            cad,
        'level':              level.get(b, '?'),
        'source_table':       src_table.get(b, '?'),
        'is_binary_indicator': b in BINARIES,
        'in_agg_means':       b in present['agg_means'],
        'in_agg_full_moments':b in present['agg_full_moments'],
        'in_panel':           b in present['panel'],
        'n_moment_cols':      len(fm_cols),
        'moment_suffixes':    ' '.join(s.lstrip('_') for s in suffixes),
        'old_name':           f'monthly_{b}' if cad == 'monthly' else b,
    })
inv = pd.DataFrame(rows)

print('\n' + '=' * 100)
print('INVENTORY BUILT')
print('=' * 100)
print(f'  rows {len(inv)}')
print('\n  by cadence and level:')
print(pd.crosstab(inv['cadence'], inv['level'], margins=True).to_string())
print('\n  moment columns per stock factor (aggregate full moments):')
print(inv[inv['level'] == 'stock']['n_moment_cols']
        .value_counts().sort_index().to_string())
print('\n  binary regime indicators, which have no theme by design:')
print('    ' + ', '.join(inv.loc[inv['is_binary_indicator'], 'base_factor']))


# ── CELL 4 : merge descriptions from an older inventory ─────────────────────
# The old pipeline prefixed monthly features and stored one row per COLUMN, so a
# stock factor appears up to six times with identical description text. Both are
# normalised before the join.
#
# _spread is the trap: thirteen macro factors legitimately end in it and must NOT
# be stripped. The exact name is therefore tried first, and stripping applies
# only where it fails to match.

def normalise_old(name, valid):
    """Old column name -> current base factor name."""
    if name in valid:
        return name
    s = name[len('monthly_'):] if name.startswith('monthly_') else name
    if s in valid:
        return s
    for suf in MOMENTS:
        if s.endswith(suf):
            t = s[:-len(suf)]
            if t in valid:
                return t
    return s


print('\n' + '=' * 100)
print('DESCRIPTIONS')
print('=' * 100)

valid = set(all_base)
desc = pd.Series(dtype=object)
extra_cols = {}

for path in DESC_CANDIDATES:
    if not path.exists():
        print(f'  not found : {path.name}')
        continue

    old = pd.read_csv(path)
    name_col = next((c for c in ('column', 'feature', 'factor', 'name',
                                 'base_factor') if c in old.columns), None)
    desc_col = next((c for c in ('description', 'desc', 'definition',
                                 'label', 'notes') if c in old.columns), None)

    print(f'\n  {path.name}: {len(old)} rows')
    print(f'    columns: {list(old.columns)}')
    if name_col is None:
        print('    !! no recognisable name column, skipped')
        continue

    old['_base'] = [normalise_old(str(n), valid) for n in old[name_col]]
    matched = old['_base'].isin(valid)
    print(f'    name column  : {name_col}')
    print(f'    desc column  : {desc_col if desc_col else "NONE FOUND"}')
    print(f'    rows mapping to a current base factor: {matched.sum()} of {len(old)}')

    keep = old[matched].drop_duplicates('_base').set_index('_base')

    if desc_col:
        new = keep[desc_col].dropna()
        added = new.index.difference(desc.index)
        desc = pd.concat([desc, new.loc[added]])
        print(f'    descriptions added: {len(added)}')

    # carry any other useful columns through, without overwriting
    for c in keep.columns:
        if c in (name_col, desc_col) or c.startswith('_'):
            continue
        if c in ('theme', 'theme_name', 'subtheme', 'subtheme_name',
                 'category', 'source', 'vendor', 'family', 'units', 'frequency'):
            if c not in extra_cols:
                extra_cols[c] = keep[c].dropna()
                print(f'    also carried: {c} ({extra_cols[c].notna().sum()} values)')

inv['description'] = inv['base_factor'].map(desc)
for c, s in extra_cols.items():
    inv[f'old_{c}'] = inv['base_factor'].map(s)

n_desc = inv['description'].notna().sum()
print(f'\n  descriptions attached: {n_desc} of {len(inv)}  ({n_desc/len(inv):.0%})')

missing = inv.loc[inv['description'].isna(), ['base_factor', 'cadence', 'level']]
if len(missing):
    print(f'\n  WITHOUT a description ({len(missing)}):')
    print(missing.to_string(index=False))
    print('\n  These are either genuinely new to this pipeline (the weekly _diff')
    print('  factors), the binary regime indicators, or a name the normaliser')
    print('  could not map. Check the last group by hand.')


# ── CELL 5 : save ───────────────────────────────────────────────────────────
inv = inv[['base_factor', 'description', 'cadence', 'level', 'source_table',
           'is_binary_indicator', 'in_agg_means', 'in_agg_full_moments',
           'in_panel', 'n_moment_cols', 'moment_suffixes', 'old_name']
          + [c for c in inv.columns if c.startswith('old_')
             and c != 'old_name']]

path = OUT_DIR / 'factor_inventory.csv'
inv.to_csv(path, index=False)

print('\n' + '=' * 100)
print('SAVED')
print('=' * 100)
print(f'  {path}')
print(f'  {len(inv)} base factors, {len(inv.columns)} columns')
print(f'\n  first 15 rows:')
print(inv.head(15).to_string(index=False, max_colwidth=44))
print(f"""
  NOTES
  -----
  Regenerated from 02_assembled, so every factor removed anywhere in the
  pipeline is absent. Add a theme_id / subtheme_id column and fill it in.

  in_agg_means, in_agg_full_moments and in_panel should all be True for every
  row, since the union enforces an identical base factor set. Any False means
  something upstream changed.

  n_moment_cols is 1 for macro factors and up to 6 for stock factors: the
  cap-weighted mean in the means table plus five cross-sectional moments in the
  full-moments table. Values below 6 mean some moments were dropped
  individually, which happened to 21 factors.

  The five binary regime indicators pass through un-z-scored and were never
  assessed by the exclusion rules, so they have no theme by design. Decide
  whether to theme them or leave them out.
""")

CADENCE AND LEVEL
  base factors catalogued : 579
  cadence : daily 290   weekly 32   monthly 257
  level   : stock 288   macro 291

BASE FACTORS PRESENT IN EACH FINAL DATASET
  agg_means            581 columns     579 features    579 base factors
  agg_full_moments    1706 columns    1704 features    579 base factors
  panel                583 columns     579 features    579 base factors

  union across all three: 579
  base factors not common to panel and agg_means: 0   (identity enforced)

INVENTORY BUILT
  rows 579

  by cadence and level:
level    macro  stock  All
cadence                   
daily      156    134  290
monthly    103    154  257
weekly      32      0   32
All        291    288  579

  moment columns per stock factor (aggregate full moments):
n_moment_cols
2      1
3      5
4     15
5    266
6      1

  binary regime indicators, which have no theme by design:
    credit_stress, curve_inverted_2y10y, curve_inverted_3m10y, vix_above_20, vix_above_30

DESCRIPTIONS

  c

In [4]:
import pandas as pd
from pathlib import Path

# 1. Define the path exactly as it is in your scripts
ROOT = Path('../../../Data/Data_Collection/Final')
INV_PATH = ROOT / 'Stage_5_Model_Ready' / '05_themes' / 'factor_inventory.csv'

# 2. Load the base factor inventory
inv = pd.read_csv(INV_PATH)

# 3. Define the missing descriptions
missing_descriptions = {
    'bank_credit_diff': 'The week-over-week change in total commercial bank credit in the U.S., derived from the Federal Reserve H.8 release.',
    'ci_loans_diff': 'The week-over-week change in U.S. commercial and industrial (C&I) bank loans outstanding, derived from the Federal Reserve H.8 release.',
    'fed_assets_diff': 'The week-over-week change in the total assets held on the Federal Reserve balance sheet, derived from the weekly H.4.1 report.',
    'open_interest_diff': 'The week-over-week change in total open interest for S&P 500 E-Mini futures contracts, derived from the weekly CFTC Commitments of Traders report.',
    'reserves_diff': 'The week-over-week change in total reserve balances maintained by commercial banks at the Federal Reserve, derived from the weekly H.4.1 report.'
}

# 4. Inject them into the dataframe
for factor, desc in missing_descriptions.items():
    inv.loc[inv['base_factor'] == factor, 'description'] = desc

# 5. Save it back to the exact same file
inv.to_csv(INV_PATH, index=False)

# Quick check to confirm it worked
remaining_missing = inv['description'].isna().sum()
print(f"Successfully updated! There are now {remaining_missing} factors without a description.")

Successfully updated! There are now 0 factors without a description.


In [8]:
import pandas as pd
from pathlib import Path

# Define paths
ROOT = Path('../../../Data/Data_Collection/Final')
INV_PATH = ROOT / 'Stage_5_Model_Ready' / '05_themes' / 'factor_inventory.csv'
THEMES_PATH = ROOT / 'Stage_3_Model_Ready' / 'themes' / 'combined_full_moments_theme_assignment.csv'

# 1. Load the current inventory
inv = pd.read_csv(INV_PATH)
valid_bases = set(inv['base_factor'])

# Helper function to map old Stage 3 names to new clean base factors
def normalise_old(name, valid):
    if name in valid:
        return name
    s = name[len('monthly_'):] if name.startswith('monthly_') else name
    if s in valid:
        return s
    for suf in ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']:
        if s.endswith(suf):
            t = s[:-len(suf)]
            if t in valid:
                return t
    return s

if THEMES_PATH.exists():
    old_themes = pd.read_csv(THEMES_PATH)
    
    # Locate the identifier column in the old file
    name_col = next((c for c in ('column', 'feature', 'factor', 'name', 'base_factor') 
                     if c in old_themes.columns), None)
    
    if name_col:
        # 2. Map old column names to your new base factors
        old_themes['_base'] = [normalise_old(str(n), valid_bases) for n in old_themes[name_col]]
        
        # 3. Filter out the "Calendar & Regime" theme
        if 'theme_name' in old_themes.columns:
            old_themes = old_themes[old_themes['theme_name'] != 'Calendar & Regime']
        
        # 4. Extract all theme-related columns (e.g., theme_id, theme_name, subtheme_id...)
        theme_cols = [c for c in old_themes.columns if 'theme' in c.lower()]
        
        # Deduplicate so we have exactly one mapping per base factor
        theme_mapping = old_themes[old_themes['_base'].isin(valid_bases)] \
                                  .drop_duplicates('_base') \
                                  .set_index('_base')[theme_cols]
        
        # 5. Merge the themes into the inventory
        # (Drop existing theme columns if you run this cell twice by accident)
        inv = inv.drop(columns=[c for c in theme_cols if c in inv.columns], errors='ignore')
        inv = inv.join(theme_mapping, on='base_factor')
        
        # 6. Save the updated inventory
        inv.to_csv(INV_PATH, index=False)
        
        # 7. Print the diagnostics and flag missing items
        matched_count = inv['theme_name'].notna().sum()
        missing_df = inv[inv['theme_name'].isna()]
        missing_factors = missing_df['base_factor'].tolist()
        
        print('\n' + '=' * 100)
        print('THEME MERGE RESULTS')
        print('=' * 100)
        print(f"  Themes successfully attached: {matched_count} of {len(inv)}")
        print(f"  Factors without a theme match (or excluded): {len(missing_factors)}\n")
        
        if missing_factors:
            print("  FLAGGED FACTORS (NO THEME):")
            # Group into chunks of 4 for cleaner console printing
            for i in range(0, len(missing_factors), 4):
                print("    " + ", ".join(missing_factors[i:i+4]))
            
            print("\n  Note: This list should exactly match your new pipeline features (like the")
            print("  weekly '_diff' columns) plus the binary regime indicators you just excluded.")
    else:
        print("  !! Could not find a recognisable name/column identifier in the themes CSV.")
else:
    print(f"  !! Themes file not found at: {THEMES_PATH}")


THEME MERGE RESULTS
  Themes successfully attached: 569 of 579
  Factors without a theme match (or excluded): 10

  FLAGGED FACTORS (NO THEME):
    bank_credit_diff, ci_loans_diff, credit_stress, curve_inverted_2y10y
    curve_inverted_3m10y, fed_assets_diff, open_interest_diff, reserves_diff
    vix_above_20, vix_above_30

  Note: This list should exactly match your new pipeline features (like the
  weekly '_diff' columns) plus the binary regime indicators you just excluded.


In [10]:
import pandas as pd
from pathlib import Path

ROOT = Path('../../../Data/Data_Collection/Final')
INV_PATH = ROOT / 'Stage_5_Model_Ready' / '05_themes' / 'factor_inventory.csv'

# Load the inventory
inv = pd.read_csv(INV_PATH)

# Define the logical theme assignments
theme_assignments = {
    'bank_credit_diff': 'Credit Conditions',
    'ci_loans_diff': 'Credit Conditions',
    'fed_assets_diff': 'Credit Conditions',
    'reserves_diff': 'Credit Conditions',
    'open_interest_diff': 'Volatility & Options'
}

# Apply the theme assignments
for factor, theme in theme_assignments.items():
    inv.loc[inv['base_factor'] == factor, 'theme_name'] = theme

# Save the updated inventory back to the file
inv.to_csv(INV_PATH, index=False)

# Print a quick verification
print('\n' + '=' * 100)
print('MANUAL THEME ASSIGNMENT COMPLETE')
print('=' * 100)
for factor, theme in theme_assignments.items():
    print(f"  - Assigned '{factor}' to '{theme}'")

remaining_missing = inv['theme_name'].isna().sum()
print(f"\n  There are now {remaining_missing} factors without a theme_name.")
if remaining_missing > 0:
    print("  (These remaining unassigned factors should just be your binary regime indicators).")


MANUAL THEME ASSIGNMENT COMPLETE
  - Assigned 'bank_credit_diff' to 'Credit Conditions'
  - Assigned 'ci_loans_diff' to 'Credit Conditions'
  - Assigned 'fed_assets_diff' to 'Credit Conditions'
  - Assigned 'reserves_diff' to 'Credit Conditions'
  - Assigned 'open_interest_diff' to 'Volatility & Options'

  There are now 5 factors without a theme_name.
  (These remaining unassigned factors should just be your binary regime indicators).


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# MOMENT-LEVEL FACTOR INVENTORY
#
# One row per surviving COLUMN (not per base factor), so it shows exactly which
# of the five moments each stock factor retained. Reads any full-moments parquet
# — 02_assembled or a split file — since the column list is identical across all
# of them and only the rows differ.
#
# Suggested location: same notebook as 02_factor_inventory.ipynb, as a second
# cell block, or standalone. Read-only apart from the one CSV it writes.
# ═══════════════════════════════════════════════════════════════════════════════

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

sys.path.append('../..')
import lib.review as rv

ROOT     = Path('../../../Data/Data_Collection/Final')
OUT_DIR  = ROOT / 'Stage_5_Model_Ready' / '05_themes'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Any of these gives an identical column list. Use whichever is already open in
# your session; the schema-only read below never loads the actual data.
FM_PATH = ROOT / 'Stage_5_Model_Ready' / '02_assembled' / 'agg_full_moments.parquet'
# FM_PATH = ROOT / 'Stage_5_Model_Ready' / '04_splits' / 'Split_A' / 'agg_full_moments_test.parquet'

MEANS_PATH = ROOT / 'Stage_5_Model_Ready' / '02_assembled' / 'agg_means.parquet'

DESC_CSV = ROOT / 'Stage_5_Model_Ready' / '05_themes' / 'factor_inventory.csv'
# the base-factor inventory built previously, with the 'description' column

META = {'date', 'target_daily_return', 'permno', 'dlyret', 'dlycap'}
BINARIES = {'vix_above_20', 'vix_above_30', 'curve_inverted_2y10y',
           'curve_inverted_3m10y', 'credit_stress'}
MOMENTS = ('_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread')


# ── CELL 1 : read columns only, no data ───────────────────────────────────────
# pq.read_schema never loads a row, so this is instant regardless of file size.

fm_cols    = pq.read_schema(FM_PATH).names
means_cols = pq.read_schema(MEANS_PATH).names

feat_cols = [c for c in fm_cols if c not in META and c not in BINARIES]

print('=' * 100)
print('MOMENT-LEVEL INVENTORY')
print('=' * 100)
print(f'  source           : {FM_PATH.name}')
print(f'  total columns     : {len(fm_cols)}')
print(f'  feature columns   : {len(feat_cols)}   '
      f'(excludes meta and {len(BINARIES)} binary regime indicators)')


# ── CELL 2 : base factor and moment, per column ───────────────────────────────
# base_factor_map is exact: it strips a moment suffix ONLY when a matching
# _cwmean column exists among the columns supplied. This is why it needs the
# full column list rather than working name-by-name, and why a naive suffix
# strip would corrupt the thirteen macro factors whose names legitimately end
# in _spread (bull_bear_spread, vix_term_spread, and so on) — none of those has
# a companion _cwmean, so the map correctly leaves them whole.

bmap = rv.base_factor_map(fm_cols)
means_set = set(means_cols) - META

rows = []
for c in feat_cols:
    base = bmap[c]
    if c == base:
        # unsuffixed: a macro raw level, OR the base factor's own means-table
        # entry appearing again here (should not happen in full_moments, but
        # checked rather than assumed)
        moment = 'raw_level'
    else:
        moment = next(s.lstrip('_') for s in MOMENTS if c.endswith(s) and
                     c[:-len(s)] == base)
    rows.append({
        'column':      c,
        'base_factor': base,
        'moment':      moment,
        'level':       'macro' if base not in
                       {bmap[x] for x in feat_cols if x.endswith('_cwmean')}
                       else 'stock',
    })

inv = pd.DataFrame(rows)

print(f'\n  columns by moment type:')
print(inv['moment'].value_counts().to_string())


# ── CELL 3 : completeness per base factor ─────────────────────────────────────
# A full stock factor has 5 moments. Fewer than 5 means some were dropped
# individually by a measured rule while the cap-weighted mean survived.

ALL_MOMENTS = [s.lstrip('_') for s in MOMENTS]

comp = (inv[inv['level'] == 'stock']
          .groupby('base_factor')['moment']
          .apply(lambda s: set(s))
          .reset_index(name='present'))
comp['n_present'] = comp['present'].apply(len)
comp['missing']   = comp['present'].apply(
    lambda s: ' '.join(m for m in ALL_MOMENTS if m not in s))
comp = comp.drop(columns='present')

print('\n' + '=' * 100)
print('MOMENT COMPLETENESS — STOCK FACTORS')
print('=' * 100)
print(f'\n  n_present distribution:')
print(comp['n_present'].value_counts().sort_index().to_string())

incomplete = comp[comp['n_present'] < 5].sort_values('n_present')
print(f'\n  {len(incomplete)} stock factors with at least one moment missing:')
print(incomplete.to_string(index=False))


# ── CELL 4 : pivot to one row per base factor, moments as columns ────────────
# The wide form is usually what you actually want for retheming: one row per
# factor, a column per moment showing whether that column exists in the final
# dataset.

wide = (inv.assign(present=True)
           .pivot_table(index=['base_factor', 'level'], columns='moment',
                        values='present', fill_value=False, aggfunc='first')
           .reset_index())

for m in ['cwmean', 'cwstd', 'cwskew', 'cwkurt', 'spread', 'raw_level']:
    if m not in wide.columns:
        wide[m] = False

order = ['base_factor', 'level', 'raw_level', 'cwmean', 'cwstd',
        'cwskew', 'cwkurt', 'spread']
wide = wide[[c for c in order if c in wide.columns]]


# ── CELL 5 : merge descriptions ──────────────────────────────────────────────
if DESC_CSV.exists():
    desc = pd.read_csv(DESC_CSV)[['base_factor', 'description', 'cadence']]
    wide = wide.merge(desc, on='base_factor', how='left')
    n = wide['description'].notna().sum()
    print(f'\n  descriptions attached: {n} of {len(wide)}')
    if n < len(wide):
        print('  without a description:')
        print('    ' + ', '.join(sorted(wide.loc[wide['description'].isna(),
                                                  'base_factor'])))
else:
    print(f'\n  {DESC_CSV.name} not found — run the base-factor inventory first, '
          f'or point DESC_CSV at wherever it landed. Saving without descriptions.')


# ── CELL 6 : save both shapes ─────────────────────────────────────────────────
long_path = OUT_DIR / 'moment_inventory_long.csv'
wide_path = OUT_DIR / 'moment_inventory_wide.csv'

inv.to_csv(long_path, index=False)
wide.to_csv(wide_path, index=False)

print('\n' + '=' * 100)
print('SAVED')
print('=' * 100)
print(f'  {long_path.name}   {len(inv)} rows   one row per surviving column')
print(f'  {wide_path.name}   {len(wide)} rows   one row per base factor, '
      f'moments as True/False columns')
print(f'\n  first 10 rows of the wide file:')
print(wide.head(10).to_string(index=False, max_colwidth=40))
print(f"""
  NOTES
  -----
  raw_level is True only for macro factors and for the thirteen macro names
  that legitimately end in _spread (bull_bear_spread etc.) — the map correctly
  leaves these whole because none has a companion _cwmean.

  For stock factors, cwmean should be True on every row: the union enforces the
  cap-weighted mean as the condition for the base factor to survive at all. If
  any stock row shows cwmean=False, something upstream has changed since the
  last assembly run and is worth investigating before retheming from this file.

  Total feature count should reconcile: 291 macro raw_level + 288 stock x
  (up to 5 moments each) = 1704 if every stock factor were complete, minus the
  21 factors with a missing moment shown in Cell 3.
""")

MOMENT-LEVEL INVENTORY
  source           : agg_full_moments.parquet
  total columns     : 1706
  feature columns   : 1699   (excludes meta and 5 binary regime indicators)

  columns by moment type:
moment
cwmean       288
raw_level    287
cwskew       283
spread       283
cwstd        279
cwkurt       279

MOMENT COMPLETENESS — STOCK FACTORS

  n_present distribution:
n_present
2      1
3      5
4     15
5    266
6      1

  21 stock factors with at least one moment missing:
              base_factor  n_present             missing
                      Tax          2 cwstd cwskew cwkurt
       ptg_median_implied          3       cwskew cwkurt
               DelBreadth          3        cwstd cwkurt
           EquityDuration          3        cwstd cwkurt
                      RoE          3        cwstd cwskew
 buynumtrades_inst50k_pct          3        cwstd cwkurt
                    ChTax          4              cwkurt
          retail_dv_share          4              cwkurt
      

In [13]:
import pandas as pd
from pathlib import Path

# 1. Define paths
ROOT = Path('../../../Data/Data_Collection/Final')
OUT_DIR = ROOT / 'Stage_5_Model_Ready' / '05_themes'

WIDE_PATH = OUT_DIR / 'moment_inventory_wide.csv'
INV_PATH = OUT_DIR / 'factor_inventory.csv'
SUMMARY_PATH = OUT_DIR / 'base_factor_theme_summary.csv'

# 2. Load the wide inventory and the base inventory (which holds the themes)
wide = pd.read_csv(WIDE_PATH)
inv = pd.read_csv(INV_PATH)

# Extract just the theme column and rename it to 'theme' to match your request
themes_to_merge = inv[['base_factor', 'theme_name']].rename(columns={'theme_name': 'theme'})

# 3. Merge the theme column into the wide inventory
# Drop it first if it already exists so we don't get duplicate 'theme_x', 'theme_y' columns
if 'theme' in wide.columns:
    wide = wide.drop(columns=['theme'])
    
wide = wide.merge(themes_to_merge, on='base_factor', how='left')

# Save the updated wide inventory
wide.to_csv(WIDE_PATH, index=False)

# 4. Create the focused 5-column subset
# Ensure we have all necessary columns available
columns_wanted = ['base_factor', 'level', 'cadence', 'theme', 'description']

# Extract just those columns
summary_df = wide[columns_wanted]

# Save the new subset CSV
summary_df.to_csv(SUMMARY_PATH, index=False)

# 5. Print confirmation
print('\n' + '=' * 100)
print('FILES UPDATED AND CREATED')
print('=' * 100)
print(f"  1. Updated: {WIDE_PATH.name} (added 'theme' column)")
print(f"  2. Created: {SUMMARY_PATH.name} (5 specific columns)")
print(f"\n  Summary file preview:")
print(summary_df.head().to_string(index=False))


FILES UPDATED AND CREATED
  1. Updated: moment_inventory_wide.csv (added 'theme' column)
  2. Created: base_factor_theme_summary.csv (5 specific columns)

  Summary file preview:
       base_factor level cadence                            theme                                               description
                AM stock monthly                        Valuation cwmean of: Assets-to-market: total assets / market equity
  AbnormalAccruals stock monthly Profitability & Earnings Quality    cwmean of: Abnormal accruals from modified Jones model
          Accruals stock monthly Profitability & Earnings Quality                  cwmean of: Total accruals / total assets
AnnouncementReturn stock monthly Profitability & Earnings Quality          cwmean of: Earnings announcement abnormal return
       AssetGrowth stock monthly Investment & Corporate Structure                         cwmean of: Total asset growth YoY


In [14]:
import pandas as pd
import re
from pathlib import Path

# 1. Define paths
ROOT = Path('../../../Data/Data_Collection/Final')
OUT_DIR = ROOT / 'Stage_5_Model_Ready' / '05_themes'
SUMMARY_PATH = OUT_DIR / 'base_factor_theme_summary.csv'

# Create the new subfolder for the split files
THEME_DIR = OUT_DIR / 'by_theme'
THEME_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load the summary file
df = pd.read_csv(SUMMARY_PATH)

# Handle the 5 binary regime indicators (which intentionally have no theme)
df['theme'] = df['theme'].fillna('Unassigned_Binary_Indicators')

# 3. Get unique themes
unique_themes = df['theme'].unique()

print('\n' + '=' * 100)
print(f'SAVING {len(unique_themes)} THEME FILES')
print('=' * 100)

# 4. Loop through each theme, filter, and save
for theme in unique_themes:
    # Sanitize the theme name so it is safe for filenames
    safe_name = theme.replace(" & ", "_and_").replace(" ", "_")
    safe_name = re.sub(r'[^A-Za-z0-9_]', '', safe_name) 
    
    file_name = f"{safe_name}_factor_inventory.csv"
    save_path = THEME_DIR / file_name
    
    # Filter for just this theme
    theme_df = df[df['theme'] == theme]
    
    # Save to the new subfolder
    theme_df.to_csv(save_path, index=False)
    
    print(f"  - Saved: {file_name:<45} ({len(theme_df):>3} factors)")

print(f"\nAll files successfully saved to: {THEME_DIR}")


SAVING 13 THEME FILES
  - Saved: Valuation_factor_inventory.csv                ( 18 factors)
  - Saved: Profitability_and_Earnings_Quality_factor_inventory.csv ( 18 factors)
  - Saved: Investment_and_Corporate_Structure_factor_inventory.csv ( 30 factors)
  - Saved: CrossSectional_Risk_Profile_factor_inventory.csv ( 20 factors)
  - Saved: Liquidity_and_Market_Quality_factor_inventory.csv ( 41 factors)
  - Saved: Analyst_Expectations_and_Sentiment_factor_inventory.csv ( 47 factors)
  - Saved: Momentum_and_Reversal_factor_inventory.csv    ( 40 factors)
  - Saved: Volatility_and_Options_factor_inventory.csv   ( 97 factors)
  - Saved: Order_Flow_and_Participation_factor_inventory.csv ( 48 factors)
  - Saved: Credit_Conditions_factor_inventory.csv        ( 25 factors)
  - Saved: Global_Markets_FX_and_Commodities_factor_inventory.csv ( 51 factors)
  - Saved: Macroeconomic_Fundamentals_factor_inventory.csv ( 81 factors)
  - Saved: Interest_Rates_and_Monetary_Policy_factor_inventory.csv ( 58 f

In [1]:
import pandas as pd
import re
from pathlib import Path

# ==============================================================================
# 1. SETUP PATHS
# ==============================================================================
ROOT = Path('../../../Data/Data_Collection/Final')
INVENTORY_PATH = ROOT / 'Stage_5_Model_Ready' / '05_themes' / 'classified_moment_inventory_long.csv'
OUT_DIR = ROOT / 'Stage_5_Model_Ready' / '05_themes' / 'shape_subthemes'

# Create the output directory if it doesn't exist
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("EXTRACTING SKEW & KURTOSIS FACTORS FOR REGROUPING")
print("=" * 80)

# ==============================================================================
# 2. LOAD AND FILTER DATA
# ==============================================================================
if not INVENTORY_PATH.exists():
    raise FileNotFoundError(f"Could not find inventory at: {INVENTORY_PATH.resolve()}")

df = pd.read_csv(INVENTORY_PATH)

# Filter for factors where the column name indicates skew or kurtosis,
# or where the exact moment type matches.
mask_shape = df['moment'].isin(['cwskew', 'cwkurt']) | df['column'].str.contains('_cwskew|_cwkurt', na=False)
shape_df = df[mask_shape].copy()

print(f"Found {len(shape_df)} shape factors (skew/kurtosis) across {shape_df['theme'].nunique()} themes.\n")

# ==============================================================================
# 3. HELPER FUNCTION FOR FILE NAMES
# ==============================================================================
def sanitize_filename(name):
    """Converts theme names like 'Volatility & Options' into 'volatility_options'"""
    if pd.isna(name):
        return "unassigned"
    # Replace non-alphanumeric characters with underscores
    clean_name = re.sub(r'[^a-zA-Z0-9]+', '_', str(name))
    return clean_name.strip('_').lower()

# ==============================================================================
# 4. GROUP BY THEME AND EXPORT CSVs
# ==============================================================================
file_count = 0

for theme, group in shape_df.groupby('theme'):
    # Sort the group by base factor and then moment for easier manual reading
    group = group.sort_values(by=['base_factor', 'moment'])
    
    # Generate clean filename
    safe_name = sanitize_filename(theme)
    out_file = OUT_DIR / f"{safe_name}_shape_factors.csv"
    
    # Export to CSV (keeping only the most useful columns for your review)
    cols_to_keep = ['column', 'base_factor', 'moment', 'theme', 'subtheme', 'description']
    group[cols_to_keep].to_csv(out_file, index=False)
    
    file_count += 1
    print(f"  ✓ Extracted {len(group):>2} factors -> {out_file.name}")

print("\n" + "-" * 80)
print(f"Success! {file_count} CSV files have been saved to: {OUT_DIR.resolve()}")

EXTRACTING SKEW & KURTOSIS FACTORS FOR REGROUPING
Found 562 shape factors (skew/kurtosis) across 9 themes.

  ✓ Extracted 83 factors -> analyst_expectations_sentiment_shape_factors.csv
  ✓ Extracted 40 factors -> cross_sectional_risk_profile_shape_factors.csv
  ✓ Extracted 60 factors -> investment_corporate_structure_shape_factors.csv
  ✓ Extracted 78 factors -> liquidity_market_quality_shape_factors.csv
  ✓ Extracted 52 factors -> momentum_reversal_shape_factors.csv
  ✓ Extracted 94 factors -> order_flow_participation_shape_factors.csv
  ✓ Extracted 31 factors -> profitability_earnings_quality_shape_factors.csv
  ✓ Extracted 30 factors -> valuation_shape_factors.csv
  ✓ Extracted 94 factors -> volatility_options_shape_factors.csv

--------------------------------------------------------------------------------
Success! 9 CSV files have been saved to: C:\Users\Henry\OneDrive\Documents\LSE\MSc\Dissertation\Project\Data\Data_Collection\Final\Stage_5_Model_Ready\05_themes\shape_subthemes
